# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Dataset Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

<https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json>

Dataset description: Structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata.get('name', ''))
print("Description:", metadata.get('description', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant schemas, every entity is uniquely referenced by its `@id`. Here we'll enumerate available record sets and their fields, referencing each by its `@id`.

In [ ]:
# List all record sets by their '@id' and print their fields
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    rs_obj = dataset.record_set(rs)
    print(f"RecordSet @id: {rs_obj.id}")
    fields = rs_obj.fields
    for fld in fields:
        print(f"    Field @id: {fld.id}, name: {fld.name}, type: {fld.data_type}")
    print()
    # Preview first record
    print(f"Sample record from RecordSet {rs_obj.id}:")
    for rec in dataset.records(record_set=rs_obj.id):
        print(rec)
        break
    print("-"*40)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the `@id` values listed above.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}

for rs_id in [rs.id for rs in map(dataset.record_set, record_sets)]:
    records = list(dataset.records(record_set=rs_id))
    print(f"RecordSet {rs_id} - {len(records)} records")
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields (columns): {df.columns.tolist()}")
        print(df.head())
    else:
        print("No records found.")
    print("-"*40)
# Example: Access one dataframe by its record set '@id'
# e.g., df = dataframes['cr:RecordSet_Table1']
# Replace with actual '@id' from previous overview

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

### Example: Remove outliers, normalize numeric fields, and group data.

We select a numeric field and group field by their `@id`. Adjust as appropriate based on the dataset loaded.

In [ ]:
# Choose one record set to work with
# (assuming the first record set has numeric fields)
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with RecordSet @id: {record_set_id}")

    # List possible numeric fields
    print("Possible numeric fields:")
    for col in df.columns:
        if df[col].dtype in [float, int]:
            print(f"  {col}")

    # Example: Use 'schema:age' if available
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        numeric_field_id = df.columns[0]  # fallback to first column

    print(f"Selected numeric field: {numeric_field_id}")

    threshold = 25
    if numeric_field_id in df.columns:
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Choose a group field (e.g., 'cr:hirsutism', 'cr:phenotype', etc.)
        group_field_id = None
        for col in df.columns:
            if 'phenotype' in col.lower() or 'hirsutism' in col.lower():
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped.head())
else:
    print("No record sets with extracted data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and scatter against another relevant field if available.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) > 0:
    df = dataframes[list(dataframes.keys())[0]]
    numeric_field_id = [col for col in df.columns if df[col].dtype in [float, int]]
    if numeric_field_id:
        plt.figure(figsize=(6,4))
        plt.hist(df[numeric_field_id[0]].dropna(), bins=10, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id[0]}")
        plt.xlabel(numeric_field_id[0])
        plt.ylabel("Frequency")
        plt.show()
    # Scatter between first two numeric fields, if possible
    if len(numeric_field_id) > 1:
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_field_id[0]], df[numeric_field_id[1]], c='coral')
        plt.xlabel(numeric_field_id[0])
        plt.ylabel(numeric_field_id[1])
        plt.title(f"{numeric_field_id[0]} vs. {numeric_field_id[1]}")
        plt.show()

## 6. Conclusion
This notebook demonstrates loading, extracting, and analyzing a clinical dataset using the Croissant standard and `mlcroissant` library.

Key findings:
- The dataset comprises multiple record sets with clear fields identified by unique `@id`s.
- Exploratory analysis and visualization can help highlight clinical feature distributions and relationships.
- The small sample size limits statistical generalization but enables structured case analysis.

For additional data modeling, consult the dataset's Croissant metadata, and always reference entities by their `@id` for clarity and reproducibility.